In [1]:
import pandas as pd
from sklearn.model_selection import TimeSeriesSplit
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
import pickle
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, Lasso
from sklearn.linear_model import ElasticNet

In [2]:
df_blogs = pd.read_csv('aggregate_blog.csv')
df_news = pd.read_csv('aggregate_news.csv')
df_socmed = pd.read_csv('aggregated_socmed.csv')

In [3]:
X_blogs = df_blogs[['year', 'month_sin', 'month_cos', 'ai_pct_lag1', 'ai_pct_lag2', 'ai_pct_ma3','time_index']]
y_blogs = df_blogs['ai_percentage']

X_news = df_news[['year', 'month_sin', 'month_cos', 'ai_pct_lag1', 'ai_pct_lag2', 'ai_pct_ma3','time_index']]
y_news = df_news['ai_percentage']

X_socmed = df_socmed[['year', 'month_sin', 'month_cos', 'ai_pct_lag1', 'ai_pct_lag2', 'ai_pct_ma3','time_index']]
y_socmed = df_socmed['ai_percentage']

In [4]:
tscv = TimeSeriesSplit(n_splits=5)
def eval_model(X, y, model, model_name):
    metrics_train = []
    metrics_test = []
    for i, (train_index, test_index) in enumerate(tscv.split(X)):
        X_train = X.iloc[train_index]
        X_test = X.iloc[test_index]
        y_train = y.iloc[train_index]
        y_test = y.iloc[test_index]
        

        
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train.values)
        X_test_scaled = scaler.transform(X_test.values)

        scaler_y = StandardScaler()
        y_train_scaled = scaler_y.fit_transform(y_train.values.reshape(-1, 1)).ravel()
        
        model.fit(X_train_scaled, y_train_scaled)
        
        y_pred_train = model.predict(X_train_scaled)
        y_pred_test = model.predict(X_test_scaled)
    
        y_pred_train = scaler_y.inverse_transform(y_pred_train.reshape(-1, 1)).ravel()
        y_pred_test = scaler_y.inverse_transform(y_pred_test.reshape(-1, 1)).ravel()
        
        train_mae = mean_absolute_error(y_train, y_pred_train)
        train_rmse = np.sqrt(mean_squared_error(y_train, y_pred_train))
        train_r2 = r2_score(y_train, y_pred_train)
        
        test_mae = mean_absolute_error(y_test, y_pred_test)
        test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
        test_r2 = r2_score(y_test, y_pred_test)


        filename = f'{model_name} scaled fold {i+1}.pkl'
        pickle.dump(model, open(filename, 'wb'))

        pickle.dump(scaler, open(f'scaler_X_{model_name}.pkl', 'wb'))
        pickle.dump(scaler_y, open(f'scaler_Y_{model_name}.pkl', 'wb'))
        
        metrics_train.append({
            'MAE': train_mae,
            'RMSE': train_rmse,
            'R2': train_r2,
        })
        metrics_test.append({
            'MAE': test_mae,
            'RMSE': test_rmse,
            'R2': test_r2
        })
    metrics_test_df = pd.DataFrame(metrics_test)
    metrics_train_df = pd.DataFrame(metrics_train)

    metrics_test_df.to_csv(f'{model_name}_test_metrics.csv', index=False)
    metrics_train_df.to_csv(f'{model_name}_train_metrics.csv', index=False)

# BLogs

In [17]:
eval_model(X_blogs, y_blogs, SVR(C=10, gamma='auto',kernel='linear'), 'svr_blogs')

In [18]:
eval_model(X_blogs, y_blogs, KNeighborsRegressor(algorithm='ball_tree', leaf_size=10, n_neighbors=5,p=2,weights='uniform'), 'knn_blogs')

In [19]:
eval_model(X_blogs, y_blogs, RandomForestRegressor(max_depth=None,max_features='log2',min_samples_split=5,min_samples_leaf=1,n_estimators=50), 'rfr_blogs')

In [20]:
eval_model(X_blogs, y_blogs, ElasticNet(alpha=0.01,l1_ratio=0.75,max_iter=10000), 'enet_blogs')

# Socmed

In [21]:
eval_model(X_socmed, y_socmed, SVR(C=10, gamma='auto',kernel='linear'), 'svr_socmed')

In [22]:
eval_model(X_socmed, y_socmed, KNeighborsRegressor(algorithm='ball_tree', leaf_size=10, n_neighbors=5,p=2,weights='uniform'), 'knn_socmed')

In [23]:
eval_model(X_socmed, y_socmed, RandomForestRegressor(max_depth=None,max_features='log2',min_samples_split=5,min_samples_leaf=1,n_estimators=50), 'rfr_socmed')

In [24]:
eval_model(X_socmed, y_socmed, ElasticNet(alpha=0.01,l1_ratio=0.75,max_iter=10000), 'enet_socmed')

# News

In [25]:
eval_model(X_news, y_news, SVR(C=10, gamma='auto',kernel='linear'), 'svr_news')

In [26]:
eval_model(X_news, y_news, KNeighborsRegressor(algorithm='ball_tree', leaf_size=10, n_neighbors=5,p=2,weights='uniform'), 'knn_news')

In [27]:
eval_model(X_news, y_news, RandomForestRegressor(max_depth=None,max_features='log2',min_samples_split=5,min_samples_leaf=1,n_estimators=50), 'rfr_news')

In [28]:
eval_model(X_news, y_news, ElasticNet(alpha=0.01,l1_ratio=0.75,max_iter=10000), 'enet_news')